In [1]:
import pandas as pd
import random
import re

In [2]:
from pathlib import Path
BASE_DIR = Path().resolve().parent
file_path = BASE_DIR / "data" / "dataset.csv"

------------------------------------------------------------
# TEMPLATES
# Structure stays clear, wording varies more
# Some are polished, some are shorter, some slightly messy
------------------------------------------------------------

In [3]:
templates = {

    # ── INTERVIEW INVITATION ──
    # Structure: greeting → impressed → invite → details → CTA → closing
    "interview_invitation": [
        # full structured
        "Hi {name},\n\nThanks for applying to the {role} role at {company}. We'd love to invite you to a {format} interview with our {job_area} team.\n\nIt should take about {duration}. Are you available on {date} at {time} or {date2} at {time2}?\n\nLet me know what works best.\n\nBest,\n{recruiter}\n{company} Recruiting",

        "Hello {name},\n\nWe reviewed your application for {role} at {company} and were impressed by your background in {job_area}.\n\nWe'd like to move you to the next step: a {format} interview with the hiring team. The conversation will be around {duration}.\n\nAvailable times:\n- {date} at {time}\n- {date2} at {time2}\n- {date3} at {time3}\n\nFeel free to suggest another time if needed.\n\nLooking forward to speaking,\n{recruiter}",

        "Dear {name},\n\nThank you for your interest in the {role} opportunity at {company}. We would like to invite you to interview with our team.\n\nThis will be a {format} interview focused on your experience in {job_area} and the role itself. Expected duration is {duration}.\n\nPlease reply with your availability for this week or next.\n\nWarm regards,\n{recruiter}\nHuman Resources, {company}",

        "Hey {name},\n\nGood news — we'd like to set up an interview for the {role} position at {company}.\n\nThis first step is a {format} chat, pretty conversational, about {duration}. We'll talk through your experience and answer any questions you have.\n\nWould {date} at {time} work? If not, send me a couple times that do.\n\nThanks!\n{recruiter}",

        "Hi {name},\n\nWanted to reach out because we'd love to interview you for the {role} role at {company}. Your background looks like a strong fit for the kind of {job_area} work this team does.\n\nIt'll be a {format} interview, around {duration}.\n\nLet me know if {date} at {time} works.\n\nBest,\n{recruiter}",

        # shorter — less structured
        "Hi {name},\n\nThanks for applying to {company}. We'd like to invite you to interview for our {role} opening.\n\nCould you do {date} at {time}? It would be a {format} conversation, about {duration}.\n\nThanks,\n{recruiter}",

        "Hello {name},\n\nWe'd like to move forward with your application for {role} at {company}.\n\nFirst interview would be {format}, around {duration}. Main goal is to talk through your experience in {job_area}.\n\nDoes {date2} at {time2} work? If not no worries — send another time.\n\nBest,\n{recruiter}",

        # minimal signal — no role/company mentioned
        "Hi {name},\n\nWe'd love to set up a call to discuss next steps. Would you be available for a {format} conversation this week?\n\nLet me know what times work for you.\n\n{recruiter}",

        "Are you free for a quick {format} call this week? We'd like to move things forward.\n\n{recruiter}",

        "Hi {name},\n\nCould you do {date} at {time} for a {format} interview? Should take about {duration}.\n\nThanks,\n{recruiter}",

        # urgent interview invitation — deadline embedded in full email
        "Hi {name},\n\nWe'd like to move forward with your application for {role} at {company}. We have a {format} interview slot available on {date} at {time}.\n\nPlease confirm as soon as possible — we're scheduling quickly and slots are limited.\n\nBest,\n{recruiter}",

        "Hi {name},\n\nGood news — we'd like to invite you to interview for the {role} role at {company}.\n\nCould you do {date} at {time} for a {format} call? Please let us know by {deadline} so we can hold your spot.\n\nThanks,\n{recruiter}",

        "Hello {name},\n\nWe reviewed your application for {role} at {company} and would love to set up an interview.\n\nWe need your availability by {deadline} — we're filling slots quickly this week.\n\nPlease reply as soon as you can.\n\n{recruiter}"
    ],

    # ── RECRUITER OUTREACH ──
    # Structure: greeting → who I am → why reaching out → opportunity → soft CTA
    "recruiter_outreach": [
        # full structured
        "Hi {name},\n\nI'm {recruiter} from {company}. I came across your profile and your experience in {job_area} stood out.\n\nWe're hiring for a {role} position and I think there may be a good fit here. Would you be open to a short {format} chat sometime this week?\n\nBest,\n{recruiter}",

        "Hello {name},\n\nI work on the recruiting team at {company}. I found your background while searching for strong candidates in {job_area}, and I wanted to reach out.\n\nWe have an open {role} role that could line up well with your experience. If you're open to it, I'd love to share more.\n\nWould a quick 15 to 20 minute call be of interest?\n\nThanks,\n{recruiter}\n{company}",

        "Hey {name},\n\nHope you're doing well. I'm reaching out from {company} because we're currently hiring a {role}. Your background looks relevant, especially your experience in {job_area}.\n\nNo pressure, but would you be open to a brief conversation?\n\nCheers,\n{recruiter}",

        "Hi {name},\n\nI came across your profile and thought I'd reach out. I'm recruiting for a {role} opening at {company}.\n\nThe team works heavily in {job_area}, and your experience looks like it could be a match. Let me know if you'd be open to learning more.\n\nBest,\n{recruiter}",

        # shorter
        "Hi {name},\n\nI'm hiring for a {role} role at {company}. Your background in {job_area} caught my eye.\n\nInterested in a quick chat?\n\n{recruiter}",

        "Hello {name},\n\nWanted to reach out from {company}. We have a {role} opening and I think your background may line up nicely with what we're looking for in {job_area}.\n\nHappy to share details or jump on a quick call.\n\nThanks,\n{recruiter}",

        # minimal signal — no role/company mentioned
        "Hi {name},\n\nI came across your profile and thought your background could be a strong fit for something we're working on. Would you be open to a quick call?\n\n{recruiter}",

        "Saw your profile — your experience looks relevant to an opening we have. Open to a short conversation?\n\n{recruiter}",

        "Hi {name},\n\nI'm recruiting for a role that looks like it could match your background. Let me know if you'd be interested in learning more.\n\nBest,\n{recruiter}",
    ],

    # ── REJECTION ──
    # Structure: greeting → thank you → decision → kind close → wish well
    "rejection": [
        # full structured
        "Hi {name},\n\nThank you again for your time and interest in the {role} position at {company}.\n\nAfter careful review, we've decided to move forward with another candidate whose background more closely matches our current needs.\n\nWe appreciate your interest and wish you the very best in your search.\n\nBest regards,\n{recruiter}\n{company}",

        "Dear {name},\n\nThank you for applying for the {role} opportunity at {company} and for the time you spent with our team.\n\nAt this time, we will not be moving forward with your candidacy. This was a competitive process and the decision was not easy.\n\nWe truly appreciate your interest and wish you success going forward.\n\nSincerely,\n{recruiter}",

        "Hi {name},\n\nI wanted to follow up regarding the {role} role at {company}.\n\nWe've decided to proceed with another candidate for this position. Thank you for taking the time to speak with us.\n\nWishing you all the best,\n{recruiter}",

        "Hello {name},\n\nThanks for your interest in {company}. We've completed our review for the {role} opening and won't be moving ahead with your application.\n\nWe appreciate the time you invested and wish you the best.\n\nWarmly,\n{recruiter}",

        # negation examples — critical for fixing offer confusion
        "Hi {name},\n\nWe are not able to extend an offer at this time. We appreciate your interest in {company} and wish you well in your search.\n\nBest,\n{recruiter}",

        "Dear {name},\n\nUnfortunately we cannot move forward with an offer for the {role} position. This was a difficult decision given the strength of our candidate pool.\n\nWe wish you all the best.\n\nSincerely,\n{recruiter}",

        "Hi {name},\n\nAfter careful consideration, we will not be extending an offer for the {role} role at {company}. Thank you for your time.\n\n{recruiter}",

        "We are unable to offer you the position at this time, but we will keep your profile on file for future openings.\n\n{recruiter}\n{company}",

        "Hi {name},\n\nWe won't be making an offer following your interviews with the {company} team. We appreciated meeting you.\n\nBest,\n{recruiter}",

        # minimal signal — no role/company mentioned
        "Thank you for your time. We've decided to move forward with another candidate.",

        "We won't be continuing with your application at this stage. Thank you for your interest.",

        "After reviewing all candidates we will not be moving forward. We wish you well.",

        "Hi {name},\n\nWe've made our decision and unfortunately won't be proceeding. Thank you for the conversations.\n\nBest,\n{recruiter}",

        # short/messy
        "Not moving forward, but thank you for your time.",

        "We went with someone else. Best of luck in your search.",
    ],

    # ── SCHEDULING ──
    # Structure: greeting → reason → new proposed time → ask for confirmation
    "scheduling": [
        # full structured
        "Hi {name},\n\nI wanted to reach out because we need to reschedule your {format} interview originally planned for {date}.\n\nWould {date2} at {time2} work instead? If not, feel free to send a few options.\n\nSorry for the inconvenience,\n{recruiter}\n{company}",

        "Hello {name},\n\nA conflict came up on our side and I need to move our interview.\n\nCould we do {date2} at {time} or {date3} at {time3} instead?\n\nAppreciate your flexibility.\n\nBest,\n{recruiter}",

        "Hi {name},\n\nSorry for the short notice, but I need to reschedule our interview for the {role} role at {company}.\n\nPlease let me know whether {date2} at {time} works. Happy to find another slot if needed.\n\nThank you,\n{recruiter}",

        "Dear {name},\n\nWe need to reschedule your upcoming interview due to an internal conflict.\n\nWe'd like to propose {date2} at {time2} as a new time. Please let us know if that works.\n\nBest regards,\n{recruiter}\n{company}",

        # booking/slot language — fixing test 13 gap
        "Hi {name},\n\nPlease use the link below to choose a time slot that works for you. We have openings on {date} and {date2}.\n\nLet me know if none of those work.\n\n{recruiter}",

        "We'd like to keep things moving — please select one of the available time slots from the calendar link.\n\n{recruiter}",

        "Hi {name},\n\nFeel free to book a slot using the scheduling link. We have availability {date} and {date2}.\n\nThanks,\n{recruiter}",

        # minimal signal — no role/company
        "Hi {name},\n\nNeed to move our call. Can you do {date2} at {time2} instead?\n\nThanks,\n{recruiter}",

        "Can we reschedule to later this week? I have a conflict on {date}.\n\n{recruiter}",

        "Do you have time {date2} instead? Something came up on my end.\n\n{recruiter}",

        # short/messy
        "Need to move our interview from {date}. Can you do {date2} at {time2}?\n\nThanks,\n{recruiter}",

        "Push our call back by a day? I have a conflict {date} morning.\n\n{recruiter}",

        "Does {date3} work instead of {date}?\n\n{recruiter}",
    ],

    # ── OFFER ──
    # Structure: greeting → congratulations → details → next steps → deadline
    "offer": [
        # full structured
        "Hi {name},\n\nI'm excited to share that we'd like to offer you the {role} position at {company}.\n\nThe offer includes a base salary of {salary}, {equity}, and our full benefits package.\n\nI'll send the formal offer letter shortly. We'd appreciate your response by {deadline}.\n\nCongratulations,\n{recruiter}\n{company}",

        "Dear {name},\n\nOn behalf of {company}, I'm pleased to extend an offer for the position of {role}.\n\nWe were impressed by your experience and believe you would be a strong addition to our {job_area} work. The compensation package includes {salary} plus benefits and {equity}.\n\nPlease review the attached letter and let us know your decision by {deadline}.\n\nWarm regards,\n{recruiter}",

        "Hello {name},\n\nGreat news — we'd love to have you join {company} as our next {role}.\n\nYour base salary would be {salary}, along with benefits and {equity}. I've attached the full offer details.\n\nWe'd love a decision by {deadline} if possible.\n\nBest,\n{recruiter}",

        "Hi {name},\n\nWe've completed the process and are happy to offer you the {role} role at {company}.\n\nThe package includes {salary}, benefits, and {equity}. Formal paperwork is on the way.\n\nPlease review everything and get back to us by {deadline}.\n\nHope to welcome you soon,\n{recruiter}",

        # shorter
        "Hi {name},\n\nExciting news — we'd like to offer you the {role} position at {company}.\n\nCompensation is {salary} plus benefits and {equity}. Offer letter coming shortly.\n\nPlease respond by {deadline}.\n\n{recruiter}",

        "Hey {name},\n\nReally happy to share this: we'd like to make you an offer for the {role} role at {company}.\n\nOffer details include {salary}, benefits, and {equity}.\n\nLet me know by {deadline}.\n\nCongrats,\n{recruiter}",

        # minimal signal
        "Hi {name},\n\nWe'd like to move forward with an offer. I'll send the formal details shortly — please respond by {deadline}.\n\nCongratulations,\n{recruiter}",

        "Offer letter is ready for your review. Please take a look and let us know your decision by {deadline}.\n\n{recruiter}",

        # urgent offer — deadline pressure embedded in full email
        "Hi {name},\n\nWe're thrilled to offer you the {role} position at {company}. Compensation is {salary} plus {equity}.\n\nThis offer expires on {deadline} — please review and respond promptly so we can get everything moving.\n\nCongratulations,\n{recruiter}",

        "Dear {name},\n\nWe're pleased to extend an offer for the {role} role at {company} with a package of {salary} and {equity}.\n\nWe need your decision by {deadline} to proceed. Please don't hesitate to reach out with any questions.\n\nWarm regards,\n{recruiter}",

        "Hi {name},\n\nOffer letter is attached for the {role} position at {company}.\n\nPlease respond by {deadline} — we need your decision to move forward with onboarding.\n\nCongrats,\n{recruiter}"
    ],

    # ── FOLLOW UP ──
    # Structure: greeting → reference → reason for following up → ask → close
    "follow_up": [
        # full structured
        "Hi {recruiter},\n\nI wanted to follow up on my application for the {role} position at {company}.\n\nI'm still very interested in the opportunity, especially given the team's work in {job_area}. Please let me know if there are any updates or if you need anything else from me.\n\nBest,\n{name}",

        "Hello {recruiter},\n\nI'm following up regarding my interview for the {role} role at {company} on {date}.\n\nI enjoyed the conversation and remain very interested. I just wanted to check whether there are any updates on timeline or next steps.\n\nThank you,\n{name}",

        "Hi {recruiter},\n\nHope you're doing well. I wanted to check in on the {role} opening at {company}.\n\nI'm still excited about the role and would appreciate any update when you have one.\n\nBest regards,\n{name}",

        "Dear {recruiter},\n\nI'm writing to follow up on my application for the {role} position at {company}.\n\nPlease let me know if there are any updates regarding the process. I'm happy to provide anything additional if helpful.\n\nSincerely,\n{name}",

        # minimal signal — no role/company
        "Hi {recruiter},\n\nJust checking in to see if there are any updates. Still very interested in the opportunity.\n\nBest,\n{name}",

        "Wanted to follow up and see where things stand. Happy to provide anything else if needed.\n\nThanks,\n{name}",

        "Any news on the timeline? I'm still very interested and just wanted to check in.\n\n{name}",

        "Hi {recruiter},\n\nFollowing up on our recent conversation. Wanted to see if there's an update on next steps.\n\nBest,\n{name}",

        # short/messy
        "Just following up. Still interested.\n\n{name}",

        "Checking in — is there a timeline yet?\n\n{name}",

        "Any updates on my application?\n\n{name}",
    ]
}

----------------------------------------------
# URGENCY OPTIONS
# Weighted choices based on category
# ML predicts this and backend decides what to do with the result
-----------------------------------------------

In [4]:
urgency_weights = {
    "interview_invitation": [("high", 0.65), ("medium", 0.35)],
    "recruiter_outreach":   [("low", 0.80), ("medium", 0.20)],
    "rejection":            [("low", 0.95), ("medium", 0.05)],
    "scheduling":           [("medium", 0.75), ("high", 0.25)],
    "offer":                [("high", 0.85), ("medium", 0.15)],
    "follow_up":            [("low", 0.70), ("medium", 0.30)]
}

In [5]:
def choose_weighted(options):
    labels  = [label  for label,  _ in options]
    weights = [weight for _, weight in options]
    return random.choices(labels, weights=weights, k=1)[0]

---------------------------------------------
# JOB FIELD DATA
# general = no job field signal in the email
---------------------------------------------

In [6]:

job_fields = {
    "software_engineering": {
        "label": "software engineering",
        "roles": [
            "Software Engineer", "Backend Engineer", "Frontend Engineer",
            "Full Stack Engineer", "Full Stack Developer",
            "Platform Engineer", "API Engineer", "Mobile Engineer",
            "DevOps Engineer", "Site Reliability Engineer",
            "Senior Software Engineer", "Staff Engineer", "Principal Engineer",
            "Junior Developer", "Software Engineer Intern", "SWE Intern",
        ],
        "companies": [
            "Google", "Meta", "Stripe", "Shopify", "Airbnb",
            "Netflix", "GitHub", "Atlassian", "Cloudflare", "Vercel",
        ],
    },
    "data_science": {
        "label": "data science and machine learning",
        "roles": [
            "Data Scientist", "Machine Learning Engineer", "ML Engineer",
            "Data Analyst", "Research Scientist", "AI Engineer",
            "Applied Scientist", "Analytics Engineer",
            "Quantitative Data Scientist", "Junior Data Analyst",
            "Data Science Intern", "ML Intern",
        ],
        "companies": [
            "OpenAI", "DeepMind", "Palantir", "Databricks",
            "Snowflake", "Scale AI", "Cohere", "Hugging Face",
        ],
    },
    "product_management": {
        "label": "product strategy and roadmap",
        "roles": [
            "Product Manager", "Senior Product Manager", "Associate Product Manager",
            "Technical Product Manager", "Group Product Manager",
            "Principal Product Manager", "Product Owner",
            "Growth Product Manager", "PM Intern",
        ],
        "companies": [
            "Apple", "Microsoft", "Notion", "Figma",
            "Linear", "Asana", "Monday.com", "Productboard",
        ],
    },
    "design": {
        "label": "product and user experience design",
        "roles": [
            "UX Designer", "Product Designer", "UI Designer",
            "Visual Designer", "Interaction Designer",
            "UX Researcher", "Design Lead", "Senior Product Designer",
            "Junior Designer", "Design Intern",
        ],
        "companies": [
            "Figma", "Adobe", "Canva", "IDEO",
            "Spotify", "Pinterest", "Dribbble", "InVision",
        ],
    },
    "marketing": {
        "label": "growth and marketing",
        "roles": [
            "Marketing Manager", "Growth Manager", "Growth Marketer",
            "Content Strategist", "Brand Manager", "SEO Specialist",
            "Performance Marketer", "Digital Marketing Specialist",
            "Social Media Manager", "Marketing Intern",
        ],
        "companies": [
            "HubSpot", "Mailchimp", "Hootsuite", "Buffer",
            "Semrush", "Marketo", "Klaviyo", "Sprout Social",
        ],
    },
    "finance": {
        "label": "finance and analysis",
        "roles": [
            "Financial Analyst", "Investment Analyst", "Risk Analyst",
            "Finance Manager", "Quantitative Analyst", "Quant Analyst",
            "Private Equity Analyst", "Associate", "Senior Analyst",
            "Finance Intern",
        ],
        "companies": [
            "Goldman Sachs", "JP Morgan", "BlackRock",
            "Citadel", "Morgan Stanley", "Fidelity", "Two Sigma",
        ],
    },
    "operations": {
        "label": "business operations and strategy",
        "roles": [
            "Operations Manager", "Business Analyst", "Strategy Analyst",
            "Program Manager", "Project Manager", "Operations Specialist",
            "Business Operations Analyst", "Strategy Associate",
            "Operations Intern",
        ],
        "companies": [
            "Amazon", "Deloitte", "McKinsey", "Accenture",
            "IBM", "Salesforce", "Oracle", "ServiceNow",
        ],
    },

    # ── general: no job field signal ──
    # empty role/company/label so templates produce signal-free emails
    "general": {
        "label": "",
        "roles": [""],
        "companies": [""],
    }
}

-----------------------
# FILLER DATA
-----------------------

In [7]:
first_names = [
    # Western
    "Sarah", "James", "Emily", "Michael", "Sophie", "David",
    # Turkish
    "Ayşe", "Mehmet", "Elif", "Emre", "Fatma", "Ilker",
    # Indian
    "Priya", "Arjun", "Ananya", "Rahul", "Neha", "Vikram",
    # Chinese
    "Wei", "Li", "Xiao", "Mei", "Jian", "Ling",
    # Korean
    "Min-jun", "Ji-woo", "Seo-yeon", "Hyun-woo", "Soo-jin", "Jae-hyun",
    # Japanese
    "Haruto", "Yuki", "Sakura", "Ren", "Aoi", "Takumi",
    # Spanish / Latin
    "Carlos", "Maria", "Juan", "Isabella", "Luis", "Camila",
    # Middle Eastern
    "Aisha", "Omar", "Layla", "Yusuf", "Zara", "Hassan",
]

recruiter_names = [
    "John", "Lisa", "Rachel", "Tom", "Nina", "Kevin", "Julia",
    "Zeynep", "Ahmet", "Seda", "Burak",
    "Ravi", "Anita", "Kiran", "Deepak",
    "Chen", "Wang", "Zhang", "Liu",
    "Kim", "Park", "Lee", "Choi",
    "Sato", "Tanaka", "Suzuki", "Yamamoto",
    "Luis", "Maria", "Jose", "Ana",
    "Ahmed", "Fatima", "Omar", "Noor",
]

dates   = ["Monday", "Tuesday", "Wednesday", "this Thursday",
           "March 10th", "March 15th", "next Monday", "April 2nd"]
dates2  = ["the following Tuesday", "March 17th", "next Friday",
           "April 5th", "the week after", "Thursday morning"]
dates3  = ["Friday afternoon", "April 8th", "next Wednesday"]
times   = ["10:00 AM", "2:00 PM", "3:30 PM", "11:00 AM", "4:00 PM", "9:30 AM"]
times2  = ["1:00 PM", "3:00 PM", "10:30 AM", "2:30 PM"]
times3  = ["11:30 AM", "4:30 PM", "9:00 AM"]
formats   = ["video", "phone", "onsite", "technical", "panel"]
durations = ["15 minutes", "20 minutes", "30 minutes", "45 minutes", "1 hour"]
salaries  = ["$85,000", "$90,000", "$95,000", "$105,000", "$110,000",
             "$120,000", "$130,000", "$140,000"]
equities  = ["stock options", "RSUs", "equity participation",
             "a competitive equity package", "performance bonuses"]
deadlines = ["Friday", "end of next week", "March 20th",
             "within 5 business days", "by next Wednesday"]

-----
# NOISE FUNCTION
# Adds realistic variation — not every email is perfectly formatted
-----

In [8]:
def maybe_add_noise(text):
    # swap greeting style
    if random.random() < 0.30:
        text = text.replace("Hello", "Hi", 1)

    # shorten phrases
    if random.random() < 0.20:
        text = text.replace("Please let me know", "Let me know", 1)
    if random.random() < 0.20:
        text = text.replace("Looking forward to speaking", "Looking forward to it", 1)
    if random.random() < 0.15:
        text = text.replace("Thank you for your time", "Thanks for your time", 1)

    # strip greeting line entirely
    if random.random() < 0.15:
        lines = text.split("\n")
        if len(lines) > 2:
            text = "\n".join(lines[2:])

    # strip closing signature
    if random.random() < 0.15:
        lines = text.strip().split("\n")
        if len(lines) > 3:
            text = "\n".join(lines[:-2])

    # lowercase entire email
    if random.random() < 0.08:
        text = text.lower()

    return text

----------------------------
# FILLing TEMPLATE PLACEHOLDERS
# general field has empty role/company/label
# — replace with empty string and clean up double spaces
----------------------------

In [9]:
def fill_template(template, name, recruiter, role, company, job_area):
    text = (template
        .replace("{name}",      name)
        .replace("{recruiter}", recruiter)
        .replace("{role}",      role)
        .replace("{company}",   company)
        .replace("{job_area}",  job_area)
        .replace("{format}",    random.choice(formats))
        .replace("{duration}",  random.choice(durations))
        .replace("{date}",      random.choice(dates))
        .replace("{date2}",     random.choice(dates2))
        .replace("{date3}",     random.choice(dates3))
        .replace("{time}",      random.choice(times))
        .replace("{time2}",     random.choice(times2))
        .replace("{time3}",     random.choice(times3))
        .replace("{salary}",    random.choice(salaries))
        .replace("{equity}",    random.choice(equities))
        .replace("{deadline}",  random.choice(deadlines))
    )

    # clean up artifacts from empty general field
    text = re.sub(r"for the role at\b", "", text)
    text = re.sub(r"for at\b", "", text)
    text = re.sub(r"join as our next\b", "join our team", text)
    text = re.sub(r"application for at\b", "application", text)
    text = re.sub(r"background in\b", "background", text)
    text = re.sub(r"\bat\b\s*", "", text)      # remove lone "at"
    text = re.sub(r"\bfor\b\s*\.", ".", text)  # remove lone "for."
    text = re.sub(r"\bin\b\s*\.", ".", text)   # remove lone "in."

    # collapse whitespace LAST — after all content fixes
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+\.", ".", text)
    text = re.sub(r"\s+,", ",", text)
    text = text.strip()

    return maybe_add_noise(text)

----------------------------------
# GENERATE DATASET
----------------------------------

In [10]:
random.seed(42)

rows = []

# standard combos — 18 examples each
examples_per_combo = 18

for category, tmpl_list in templates.items():
    for job_field, field_data in job_fields.items():

        # skip general for categories where no-signal is rare
        if job_field == "general" and category in ["offer"]:
            n = 10  # fewer — offers almost always mention role/salary
        elif job_field == "general":
            n = 20  # more — rejections/followups/scheduling often have no signal
        else:
            n = examples_per_combo

        for _ in range(n):
            template  = random.choice(tmpl_list)
            name      = random.choice(first_names)
            recruiter = random.choice(recruiter_names)
            role      = random.choice(field_data["roles"])
            company   = random.choice(field_data["companies"])
            job_area  = field_data["label"]
            urgency   = choose_weighted(urgency_weights[category])

            text = fill_template(template, name, recruiter, role, company, job_area)

            rows.append({
                "text":      text,
                "category":  category,
                "urgency":   urgency,
                "job_field": job_field,
            })

random.shuffle(rows)

df = pd.DataFrame(rows)
df.to_csv(file_path, index=False)

print(f"\nDataset created: {df.shape} rows/columns")
print("─" * 45)
print("\n→ Category distribution:")
print(df["category"].value_counts())
print("\n→ Urgency distribution:")
print(df["urgency"].value_counts())
print("\n→ Job field distribution:")
print(df["job_field"].value_counts())
print("\n→ General field breakdown by category:")
print(df[df["job_field"] == "general"]["category"].value_counts())
print("\n─── Preview ───")
print(df.head())



Dataset created: (866, 4) rows/columns
─────────────────────────────────────────────

→ Category distribution:
category
rejection               146
interview_invitation    146
scheduling              146
recruiter_outreach      146
follow_up               146
offer                   136
Name: count, dtype: int64

→ Urgency distribution:
urgency
low       354
medium    261
high      251
Name: count, dtype: int64

→ Job field distribution:
job_field
general                 110
design                  108
software_engineering    108
data_science            108
finance                 108
product_management      108
marketing               108
operations              108
Name: count, dtype: int64

→ General field breakdown by category:
category
follow_up               20
recruiter_outreach      20
interview_invitation    20
rejection               20
scheduling              20
offer                   10
Name: count, dtype: int64

─── Preview ───
                                           